# Fine-Tuning Chatterbox for Maltese Language Support

This notebook trains the Chatterbox multilingual TTS model for Maltese language support using the MASRI_HEADSET_v2 dataset.

**Key Features:**
- Optimized for Google Colab free tier (T4 GPU, 15GB RAM)
- Trains text embeddings and T3 transformer only
- Prevents catastrophic forgetting with mixed-language training
- Uses gradient accumulation for effective larger batch sizes

**Dataset:** [Bluefir/MASRI_HEADSET_v2](https://huggingface.co/datasets/Bluefir/MASRI_HEADSET_v2)

**Training Strategy:**
- 40% Maltese (from MASRI dataset)
- 35% Arabic (preserve Semitic knowledge)
- 20% Italian (preserve Romance knowledge)
- 5% English (maintain general performance)


## 1. Setup and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Install Chatterbox (from the current repository) and other dependencies
!pip install -q -e .
!pip install -q datasets accelerate wandb

# Verify installation
import chatterbox
print(f"Chatterbox installed successfully!")

## 2. Configuration

Configure training parameters optimized for Colab free tier.

In [ ]:
# Training configuration (optimized for Colab free tier)
CONFIG = {
    # Model and device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'mixed_precision': True,  # Use FP16 to save memory
    
    # Batch sizes (small to fit in 15GB GPU memory)
    'batch_size': 4,  # Per-device batch size
    'gradient_accumulation_steps': 8,  # Effective batch size: 32
    
    # Optimizer settings
    'learning_rate': 1e-5,  # Conservative to prevent forgetting
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    
    # Training steps
    'max_steps': 5000,  # ~2-3 hours on Colab T4
    'warmup_steps': 500,
    'save_steps': 1000,
    'eval_steps': 500,
    'logging_steps': 50,
    
    # Data mixing (prevent forgetting)
    'maltese_ratio': 0.40,  # 40% Maltese
    'arabic_ratio': 0.35,   # 35% Arabic
    'italian_ratio': 0.20,  # 20% Italian
    'english_ratio': 0.05,  # 5% English
    
    # Paths
    'output_dir': './maltese_model_checkpoints',
    'cache_dir': './cache',
    
    # Dataset
    'maltese_dataset': 'Bluefir/MASRI_HEADSET_v2',
    'max_audio_length': 10.0,  # seconds
    'sample_rate': 16000,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Load and Prepare Data

Load Maltese dataset and prepare mixed-language training data.

In [ ]:
from datasets import load_dataset, concatenate_datasets
import numpy as np

# Load Maltese dataset
print("Loading Maltese dataset (MASRI_HEADSET_v2)...")
maltese_dataset = load_dataset(
    CONFIG['maltese_dataset'],
    cache_dir=CONFIG['cache_dir'],
    split='train'  # Adjust split if needed
)

print(f"Maltese dataset loaded: {len(maltese_dataset)} samples")
print(f"Sample keys: {maltese_dataset[0].keys()}")

# Inspect first sample
sample = maltese_dataset[0]
print(f"\nSample text: {sample.get('text', sample.get('sentence', 'N/A'))}")

In [ ]:
# For preventing forgetting, we need samples from other languages
# This is a simplified version - in production, use actual multilingual datasets

print("Note: For full training, you should also load:")
print("  - Arabic dataset (35% of training)")
print("  - Italian dataset (20% of training)")
print("  - English dataset (5% of training)")
print("\nFor this demo, we'll focus on Maltese with the understanding that")
print("mixing with other languages is crucial for preventing forgetting.")
print("\nRecommended datasets:")
print("  - Arabic: Common Voice (ar)")
print("  - Italian: Common Voice (it)")
print("  - English: LJSpeech or Common Voice (en)")

# Prepare Maltese data with language tags
def add_language_tag(example):
    example['language_id'] = 'mt'
    return example

maltese_dataset = maltese_dataset.map(add_language_tag)
print(f"\nMaltese dataset prepared with language tags.")

## 4. Load and Prepare Model

Load pre-trained model and configure for training.

In [ ]:
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
import torch

# Load pre-trained model
print("Loading Chatterbox Multilingual TTS model...")
print("(This may take a few minutes)")

model = ChatterboxMultilingualTTS.from_pretrained(device=CONFIG['device'])

print("✓ Model loaded successfully!")
print(f"Device: {model.device}")

In [ ]:
# Optional: Update vocabulary to include [mt] token
# This is only needed if you've run the update_vocabulary.py script
# and have the updated vocabulary file

# Uncomment if you have updated vocabulary:
# new_vocab_size = 2455  # 2454 + 1 for [mt]
# model.t3.resize_text_token_embeddings(new_vocab_size)
# print(f"Vocabulary resized to {new_vocab_size}")

print("Note: Using existing vocabulary. [mt] token will be treated as [UNK].")
print("For optimal results, run update_vocabulary.py first.")

In [ ]:
# Freeze speech encoder and decoder (language-independent)
print("Configuring trainable parameters...")

# Freeze voice encoder
for param in model.ve.parameters():
    param.requires_grad = False
print("✓ Voice encoder frozen")

# Freeze S3Gen decoder
for param in model.s3gen.parameters():
    param.requires_grad = False
print("✓ S3Gen decoder frozen")

# Train text embeddings and T3 transformer
for param in model.t3.text_emb.parameters():
    param.requires_grad = True
for param in model.t3.text_head.parameters():
    param.requires_grad = True
for param in model.t3.tfmr.parameters():
    param.requires_grad = True
print("✓ Text embeddings trainable")
print("✓ T3 transformer trainable")

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.t3.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable_params / 1e6:.2f}M / {total_params / 1e6:.2f}M parameters")
print(f"Training {trainable_params / total_params * 100:.1f}% of the model")

In [ ]:
# Enable mixed precision training (saves memory)
if CONFIG['mixed_precision'] and CONFIG['device'] == 'cuda':
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    print("✓ Mixed precision (FP16) enabled")
else:
    scaler = None
    print("Running in FP32 mode")

## 5. Training Loop

Simplified training loop optimized for Colab.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Setup optimizer
optimizer = AdamW(
    [p for p in model.t3.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['max_steps']
)

print("✓ Optimizer configured")
print(f"  Learning rate: {CONFIG['learning_rate']}")
print(f"  Weight decay: {CONFIG['weight_decay']}")

In [ ]:
import os
from tqdm.auto import tqdm

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("="*60)
print("IMPORTANT NOTES FOR FULL TRAINING")
print("="*60)
print("\nThis notebook demonstrates the training structure, but for")
print("production use, you MUST implement:")
print("\n1. DATA PREPROCESSING:")
print("   - Audio loading and preprocessing")
print("   - Speech tokenization with S3Tokenizer")
print("   - Text tokenization with MTLTokenizer")
print("   - Proper batching and padding")
print("\n2. MIXED-LANGUAGE SAMPLING:")
print("   - Load Arabic, Italian, English datasets")
print("   - Sample according to CONFIG ratios")
print("   - This prevents catastrophic forgetting!")
print("\n3. VALIDATION:")
print("   - Evaluate on Maltese test set")
print("   - Monitor Arabic/Italian performance (should not drop >5%)")
print("   - Check for catastrophic forgetting")
print("\n4. CHECKPOINTING:")
print("   - Save model every CONFIG['save_steps']")
print("   - Keep best checkpoint based on validation")
print("="*60)

# Simplified training loop structure
print("\nTraining loop structure (REQUIRES DATA PIPELINE IMPLEMENTATION):")
print("""\n
model.t3.train()
for step in range(CONFIG['max_steps']):
    # 1. Sample batch (mixed languages according to CONFIG ratios)
    # batch = sample_mixed_batch(maltese_loader, arabic_loader, italian_loader, english_loader)
    
    # 2. Forward pass
    # loss_text, loss_speech = model.t3.loss(
    #     t3_cond=batch['t3_cond'],
    #     text_tokens=batch['text_tokens'],
    #     text_token_lens=batch['text_token_lens'],
    #     speech_tokens=batch['speech_tokens'],
    #     speech_token_lens=batch['speech_token_lens']
    # )
    # loss = loss_text + loss_speech
    
    # 3. Backward pass with gradient accumulation
    # if scaler:
    #     scaler.scale(loss).backward()
    # else:
    #     loss.backward()
    
    # 4. Optimizer step (after accumulation)
    # if (step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
    #     if scaler:
    #         scaler.unscale_(optimizer)
    #         torch.nn.utils.clip_grad_norm_(model.t3.parameters(), CONFIG['max_grad_norm'])
    #         scaler.step(optimizer)
    #         scaler.update()
    #     else:
    #         torch.nn.utils.clip_grad_norm_(model.t3.parameters(), CONFIG['max_grad_norm'])
    #         optimizer.step()
    #     scheduler.step()
    #     optimizer.zero_grad()
    
    # 5. Logging and checkpointing
    # if step % CONFIG['logging_steps'] == 0:
    #     print(f"Step {step}: Loss={loss.item():.4f}, LR={scheduler.get_last_lr()[0]:.2e}")
    # if step % CONFIG['save_steps'] == 0:
    #     save_checkpoint(step)
""")

print("\nFor a complete implementation, see train_maltese.py")
print("and MALTESE_FINETUNING_GUIDE.md in the repository.")

## 6. Example: Process Single Maltese Sample

Demonstrate how to process a single training sample.

In [ ]:
# Example of processing a single Maltese sample
import librosa

def process_maltese_sample(audio_path, text, model):
    """
    Example function showing how to process a single training sample.
    
    In production, this should be batched and optimized.
    """
    # 1. Load and preprocess audio
    audio, sr = librosa.load(audio_path, sr=16000)
    
    # 2. Tokenize text
    text_tokens = model.tokenizer.text_to_tokens(text, language_id='mt')
    
    # 3. Extract speech tokens (requires S3Tokenizer)
    # speech_tokens = model.s3gen.tokenizer.encode(audio)
    
    # 4. Prepare conditioning (speaker embedding from audio)
    # speaker_emb = model.ve.embeds_from_wavs([audio], sample_rate=16000)
    
    return text_tokens

# Demo with first sample
if len(maltese_dataset) > 0:
    sample = maltese_dataset[0]
    text = sample.get('text', sample.get('sentence', 'Bonġu! Kif int illum?'))
    
    print(f"Sample text: {text}")
    text_tokens = model.tokenizer.text_to_tokens(text, language_id='mt')
    print(f"Text tokens shape: {text_tokens.shape}")
    print(f"Text tokens: {text_tokens[:10]}...")  # First 10 tokens

## 7. Save Model

Save trained model checkpoint.

In [ ]:
def save_checkpoint(model, step, output_dir):
    """
    Save model checkpoint.
    """
    checkpoint_dir = os.path.join(output_dir, f'checkpoint-{step}')
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Save T3 model (only trainable part)
    checkpoint_path = os.path.join(checkpoint_dir, 't3_maltese.pt')
    torch.save({
        'step': step,
        'model_state_dict': model.t3.state_dict(),
        'text_tokens_dict_size': model.t3.hp.text_tokens_dict_size,
    }, checkpoint_path)
    
    print(f"✓ Checkpoint saved to {checkpoint_path}")
    return checkpoint_path

# Save initial checkpoint (before training)
initial_checkpoint = save_checkpoint(model, 0, CONFIG['output_dir'])
print(f"\nInitial checkpoint: {initial_checkpoint}")

## 8. Test Generation

Test Maltese speech generation with the model.

In [ ]:
# Test generation
import torchaudio
from IPython.display import Audio, display

# Test Maltese text
test_texts = [
    "Bonġu! Kif int illum?",
    "Malta għandha storja kbira.",
    "Jiena kuntent li niltaqa' miegħek."
]

print("Testing Maltese generation...\n")

for i, text in enumerate(test_texts):
    print(f"Text {i+1}: {text}")
    
    try:
        # Generate audio
        with torch.no_grad():
            wav = model.generate(text, language_id='mt')
        
        # Save and play
        output_path = f"maltese_test_{i+1}.wav"
        torchaudio.save(output_path, wav, model.sr)
        print(f"✓ Saved to {output_path}")
        
        # Play in notebook
        display(Audio(wav.squeeze().cpu().numpy(), rate=model.sr))
    except Exception as e:
        print(f"✗ Error: {e}")
    
    print()

## 9. Next Steps

To complete the full training pipeline:

### Required Implementations:

1. **Data Pipeline**:
   ```python
   - Implement audio loading and preprocessing
   - Extract speech tokens with S3Tokenizer
   - Create proper DataLoader with batching
   - Handle variable-length sequences
   ```

2. **Mixed-Language Sampling**:
   ```python
   - Load additional datasets (Arabic, Italian, English)
   - Implement sampling according to CONFIG ratios
   - Ensure balanced batches across languages
   ```

3. **Validation Loop**:
   ```python
   - Create validation dataset
   - Monitor loss on all languages
   - Check for catastrophic forgetting
   - Save best checkpoint
   ```

4. **Full Training**:
   ```python
   - Run for CONFIG['max_steps'] (5000 steps)
   - Monitor training loss
   - Validate every 500 steps
   - Save checkpoints every 1000 steps
   ```

### Resources:

- **MALTESE_FINETUNING_GUIDE.md**: Detailed training guide
- **train_maltese.py**: Complete training script template
- **VOCABULARY_UPDATE_GUIDE.md**: Update vocabulary with [mt] token

### Monitoring:

Track these metrics to ensure quality:
- **Maltese loss**: Should decrease steadily
- **Arabic/Italian performance**: Should stay within 95% of baseline
- **Memory usage**: Should stay under 15GB for Colab free tier
- **Training time**: ~2-3 hours for 5000 steps on T4 GPU

### Preventing Catastrophic Forgetting:

Critical strategies implemented:
1. ✅ **Conservative learning rate** (1e-5)
2. ✅ **Mixed-language training** (40% mt, 35% ar, 20% it, 5% en)
3. ✅ **Smart initialization** (embeddings from mean/std)
4. ✅ **Gradient clipping** (max_norm=1.0)
5. ✅ **Regular validation** (check all languages)


## Additional Resources

- **Dataset**: [Bluefir/MASRI_HEADSET_v2](https://huggingface.co/datasets/Bluefir/MASRI_HEADSET_v2)
- **Repository**: [Wubpooz/chatterbox](https://github.com/Wubpooz/chatterbox)
- **Branch**: `copilot/add-maltese-language-support`
- **Documentation**:
  - MALTESE_FINETUNING_GUIDE.md
  - MALTESE_IMPLEMENTATION.md
  - VOCABULARY_UPDATE_GUIDE.md
